# <font color=firebrick>Econometrics — M2 MBFA</font>
## Session 6 / 8 (CM) — Instrumental Variables

**Prerequisites:** Sessions 1-5 - OLS, multivariate regression, omitted variable bias, and panel data.

**Format.** This is the most challenging session of the course from a mathematical standpoint: take your time to thoroughly understand the two first Sections before moving on.

**Today's objectives:**
- Understand *endogeneity*: when $x$ and $\varepsilon$ are correlated, and why no amount of additional data fixes it.
- Understand what an **instrument** is, and the two conditions it must satisfy.
- Derive the instrumental-variables (IV) estimator, and its generalization, two-stage least squares (2SLS).
- See a classic application: does education really raise wages?
- [Bonus] What "ANOVA" and "ANCOVA" are, in the language you already know.

---


# 1. The instrumental variable

##### 1.1 When adding control variables fails to fix OLS errors

*Two stories behind one correlation*

Does higher education lead to higher wages? In any dataset, more educated workers earn more on average: the *correlation* is not in doubt. What is in doubt is *why*. Two very different causal mechanisms produce exactly the same correlation:

- **Story A: education causes wages.** School teaches students skills that employers are willing to pay for. Consider a given employee: if he or she is given the opportunity to pursue an additional year of study, *all else being equal*, his or her salary increases. The magnitude of this increase corresponds to the causal effect $\beta_1$ that we'd like to measure.
- **Story B: ability causes both.** Some students have greater *abilities* (cognitive skills, motivation, discipline, family support, etc.). These abilities make studying easier and more rewarding; as a result, these students continue their studies for longer. They *also* make these students more productive at work, which allows them to earn more, and they would earn more **even with the same level of education**. In this scenario, education and wages move in parallel because they share a common cause, not because one determines the other.

```
   ability  (unobserved)
    │            │
    ▼            ▼
   educ ───────▶ wage
          β₁
```

In reality, both mechanisms are probably at work. The trouble is that the data cannot tell them apart. When we compare a worker with 16 years of schooling to one with 12, we are comparing people who differ in education **and**, on average, in ability. The wage gap between them mixes the two effects and OLS has no way of separating them: it attributes the entire gap to education. To see how serious this is, imagine that Story B alone were true ($\beta_1=0$): OLS would *still* find a positive, highly significant effect of education on wages.

The problem is not new. This is the omitted variable bias of Session 3, seen from another angle. To understand what is going on, write the model we would *like* to estimate, with ability included:
$$
\log w_i = \beta_0+\beta_1\,\text{educ}_i + \gamma\,\text{abil}_i + v_i,
$$
where $\gamma>0$ is the direct effect of ability on wages (Story B's right-hand arrow) and $v_i$ collects everything else, unrelated to education and ability. If ability were in our dataset, Session 3 would tell us what to do: include it as a control and we are done. But it is not observed, so the regression we actually run is
$$
\log w_i = \beta_0+\beta_1\,\text{educ}_i + \varepsilon_i, \qquad\text{with}\qquad \varepsilon_i = \gamma\,\text{abil}_i + v_i.
$$
Ability has not disappeared: it just has moved into the error term. And since education and ability are correlated (Story B's left-hand arrow), the error is now correlated with the regressor:
$$
\text{Cov}(\text{educ}_i,\varepsilon_i) = \gamma\,\text{Cov}(\text{educ}_i,\text{abil}_i) \neq 0.
$$
Plugging this into the decomposition of $\hat\beta$ below gives:
$$
\hat\beta_1 \to \beta_1 + \gamma\,\dfrac{\text{Cov}(\text{educ},\text{abil})}{\text{Var}(\text{educ})}.
$$

We have exactly the omitted variable bias formula of Session 3 (effect of the omitted variable on $y$, times its association with the included regressor). Here both terms are positive, so OLS *overstates* the return to education.

The general name for this situation is **endogeneity**: the regressor $x_i$ is correlated with the error, $\text{Cov}(x_i,\varepsilon_i)\neq0$. Omitted variable bias is one case of it: Session 3 looked at *what is missing* from the regression; here we look at *where it went* (into $\varepsilon_i$). What changes compared with Session 3 is the remedy. Including the missing variable requires observing it, and "ability" is, by definition, measured nowhere. Session 5 handled a similar problem (unobserved, time-invariant heterogeneity $a_i$) by using *repeated observations of the same individual*, a structural feature of the data. Here we have no such structure (one observation per worker), so we need a fundamentally different approach.

*What $\text{Cov}(x_i,\varepsilon_i)\neq0$ means*

Recall what $\varepsilon_i$ is: *everything that affects $y_i$ other than $x_i$* (ability, health, luck, the local labour market...). OLS does not need these factors to be small; it needs them to be **unrelated to $x_i$**: knowing a worker's $x_i$ should tell you nothing about their $\varepsilon_i$. This is the zero conditional mean assumption, $E[\varepsilon_i\mid x_i]=0$, that underlies every result since Session 2. It is what justifies the *ceteris paribus* reading of $\hat\beta$: when we compare two values of $x$, the other factors are, on average, the same on both sides, so any difference in $y$ can be credited to $x$.

$\text{Cov}(x_i,\varepsilon_i)\neq0$ says the opposite: when $x_i$ is high, $\varepsilon_i$ tends to be systematically high (or systematically low). Comparing workers with different $x$ then also means comparing workers with different $\varepsilon$, and OLS credits to $x$ the part of $\varepsilon$ that moves with it. Formally, return to the decomposition from Session 2:
$$
\hat\beta = \beta + \frac{\sum_i x_i\varepsilon_i/N}{\sum_i x_i^2/N} \;\xrightarrow{N\to\infty}\; \beta + \frac{\text{Cov}(x_i,\varepsilon_i)}{\text{Var}(x_i)}.
$$
The consequences:

- **Bias that does not go away.** When $\text{Cov}(x_i,\varepsilon_i)=0$, the second term vanishes as $N$ grows and $\hat\beta$ converges to the truth. When $\text{Cov}(x_i,\varepsilon_i)\neq0$, it does **not** vanish, however large $N$ is: $\hat\beta$ converges to the *wrong* number (it is *inconsistent*). "Collect more data" fixes imprecision (Session 2's variance formula shrinks with $N$), which is a *completely different problem* from bias. More data makes a biased estimator more precisely wrong, not right.
- **The direction of the error is known, its size is not.** The bias has the sign of $\text{Cov}(x_i,\varepsilon_i)$: positive in the ability example, so OLS overestimates. But without observing $\varepsilon_i$, we cannot know by how much.
- **Inference is wrong too.** Standard errors, $t$-statistics and confidence intervals are all centred on the biased value. With a large sample, you get a narrow confidence interval around the wrong number: "statistically significant" does not mean "causal".
- **It cannot be detected from the OLS residuals.** By construction (the OLS first-order conditions), the residuals $\hat\varepsilon_i$ are *exactly* uncorrelated with $x_i$, whatever the true $\varepsilon_i$ looks like. Checking $\text{corr}(x,\hat\varepsilon)$ will always return $0$. Endogeneity is a question about the economics of the problem, not something the regression output can reveal.

We need a different estimator entirely, not a bigger sample.

Beyond unobserved confounders such as ability, two other common sources of endogeneity are worth naming:
- **measurement error** in $x$: if we observe $x_i^{obs}=x_i+e_i$ instead of the true $x_i$, the measurement error $e_i$ ends up both in the regressor and (with the opposite sign) in the error term, which mechanically correlates them;
- **simultaneity**: e.g. prices and quantities are jointly determined by supply and demand, so $x$ causes $y$ *and* $y$ causes $x$; any shock to $y$ (in $\varepsilon$) then feeds back into $x$.

All three break $\text{Cov}(x_i,\varepsilon_i)=0$ in the same way, and all three call for the same fix.

##### 1.2 The instrumental variable idea

Start from the intuition. The variation in education across workers has many sources. Some of it is "bad" variation, correlated with $\varepsilon$: some people study longer *because* they are more able. But some of it is "good" variation, unrelated to $\varepsilon$: some people study longer because a college happened to be nearby, which made it cheaper and easier to attend. If we could isolate the good variation, and look at how wages respond **only to that part** of education, the ability problem would vanish: within that part, more and less educated workers would not differ in ability on average.

An **instrument** is a variable $z_i$ that generates such good variation: a lever that pushes $x_i$ up or down for reasons unrelated to $\varepsilon_i$. We then watch whether $y_i$ moves when the lever moves $x_i$. For this to work, $z_i$ must satisfy two conditions:

1. **Relevance**: $z_i$ is correlated with the endogenous regressor, $\text{Cov}(z_i,x_i)\neq0$.

   The lever must actually be connected to $x$. IV learns about $\beta_1$ *only* from the variation in $x$ that $z$ produces, so if $z$ does not move $x$, there is nothing to learn: workers with high and low $z$ would have the same education, and any wage difference between them could not be credited to education. Concretely, the IV estimate (derived in Section 2) is a ratio:
   $$
   \beta_1 = \frac{\text{effect of } z \text{ on } y}{\text{effect of } z \text{ on } x}.
   $$
   For instance, if growing up near a college raises schooling by $0.3$ years on average, and raises wages by $3\%$ on average, then each year of schooling *induced by the lever* is worth $3\%/0.3 = 10\%$ of wage. The denominator is precisely what relevance requires to be non-zero. If it is zero, the ratio is undefined; if it is merely small (a **weak instrument**), we divide by a small, noisily estimated number and the estimate becomes wildly imprecise (Section 3). Two practical remarks: relevance must hold *after* controlling for the other regressors ($z$ must move $x$ beyond what the controls already explain), and it can be checked directly, by regressing $x$ on $z$ and the controls (the **first stage**).

2. **Exogeneity (the exclusion restriction)**: $z_i$ is uncorrelated with the error, $\text{Cov}(z_i,\varepsilon_i)=0$.

   $z_i$ affects $y_i$ **only through** $x_i$. This fails in two ways: if $z$ has a *direct* effect on $y$ (a path $z\to y$ that bypasses $x$), or if $z$ is correlated with the unobserved factors in $\varepsilon$ (a common cause of $z$ and $y$). In either case, the "effect of $z$ on $y$" in the numerator no longer comes only through $x$, and the ratio no longer measures $\beta_1$.

```
              ability, family background...   (inside ε)
                     │                │
                     ▼                ▼
  nearc4 ─────────▶ educ ──────────▶ wage
          relevance          β₁

     ✗  no arrow  nearc4 ──▶ wage          (exclusion restriction)
     ✗  no link   nearc4 ◀─▶ ability...    (exogeneity)
```

Condition 1 is checkable directly in the data (it is just a correlation, conditional on the controls). Condition 2 is not, since $\varepsilon_i$ is never observed: it must be argued for, using institutional knowledge or a design that makes $z_i$ "as good as random" with respect to whatever drives $y_i$. This is the hard, often contested part of any IV application: finding a convincing instrument is usually harder than the algebra that follows.

**A classic example (Card, 1995).** To study the return to education, use $z_i=$ "grew up near a four-year college" as an instrument for $x_i=$ years of schooling. Growing up near a college plausibly makes attending more likely: it is cheaper and more convenient (relevance). And there is no obvious direct reason why geographic proximity to a college would affect *wages* except *through* the extra schooling it encourages (exclusion). The exogeneity argument is debatable: families living near colleges tend to live in cities, where wages are higher anyway, which is a common cause of $z$ and $y$. This is why Card controls for region and urban residence, and why the argument remains a starting point rather than a proof. We use exactly this example in Section 4.

---

# 2. The IV estimator

##### 2.1 The simplest case: comparing two groups

*The intuition, with real numbers*

Take the Card data (Section 4) and split the $3\,010$ workers into two groups according to the instrument: those who grew up near a four-year college ($z_i=1$, $2\,053$ workers) and those who did not ($z_i=0$, $957$ workers). Compare the averages:

| | $z=0$ (far from a college) | $z=1$ (near a college) | Gap |
|---|---|---|---|
| Years of schooling, $\bar x$ | $12.70$ | $13.53$ | $+0.83$ year |
| Log wage, $\bar y$ | $6.155$ | $6.311$ | $+0.156$ (about $+16\%$) |

Workers who grew up near a college have $0.83$ more years of schooling, and earn about $16\%$ more. Now use the two conditions of Section 1.2:

- **Exogeneity** says the two groups do not differ in ability (or in anything else in $\varepsilon$) on average: being born near a college or not is, with respect to wages, "as good as random". So the $16\%$ wage gap is *not* an ability gap.
- **Exclusion** says that proximity affects wages *only through* schooling. So the entire $16\%$ wage gap must come from the $0.83$ extra years.

If $0.83$ years produce $15.6$ log points of wage, one year produces
$$
\hat\beta_1^{IV}=\frac{\bar y_1-\bar y_0}{\bar x_1-\bar x_0}=\frac{0.156}{0.829}\approx 0.19,
$$
i.e. roughly $19\%$ per year of schooling. This is the **Wald estimator**: the gap in outcomes between the two groups, divided by the gap in the treatment between the two groups.

Compare with what OLS does. OLS compares workers with *different education*, and these workers differ in ability: the comparison is contaminated (Section 1.1). IV compares workers with *different values of $z$*, who, by assumption, do not differ in ability. Education differs between the two groups only because the lever pushed it, and that is exactly the "good variation" we were looking for. (The OLS slope on the same data, without controls, is $0.052$; the gap with $0.19$ is discussed in Section 4.)

##### 2.2 The general formula: changing the moment condition

*OLS was already a "moment" estimator*

To generalize the Wald estimator to any $z$ (not only a binary one), it helps to see where the OLS formula comes from. Take the covariance of both sides of $y_i=\beta_0+\beta_1x_i+\varepsilon_i$ with $x_i$:
$$
\text{Cov}(x_i,y_i) = \beta_1\,\text{Var}(x_i) + \text{Cov}(x_i,\varepsilon_i).
$$
OLS *assumes* $\text{Cov}(x_i,\varepsilon_i)=0$, drops the last term and solves: $\beta_1=\text{Cov}(x,y)/\text{Var}(x)$, which is the familiar OLS slope. The whole problem of Section 1 is that this assumption is false.

*IV uses another, more credible, condition*

Do exactly the same thing, but take the covariance with $z_i$ instead of $x_i$:
$$
\underbrace{\text{Cov}(z_i,y_i)}_{\text{observed}} = \beta_1\,\underbrace{\text{Cov}(z_i,x_i)}_{\text{observed, }\neq0\text{ (relevance)}} + \underbrace{\text{Cov}(z_i,\varepsilon_i)}_{=0\text{ (exogeneity)}}.
$$
Each condition of Section 1.2 has a precise job here. Exogeneity removes the last term; relevance makes it possible to divide by $\text{Cov}(z_i,x_i)$. Solving for $\beta_1$, and replacing population covariances by their sample counterparts, gives the **IV estimator**:
$$
\boxed{\;\hat\beta_1^{IV} = \frac{\sum_i(z_i-\bar z)(y_i-\bar y)}{\sum_i(z_i-\bar z)(x_i-\bar x)} = \frac{\widehat{\text{Cov}}(z,y)}{\widehat{\text{Cov}}(z,x)}.\;}
$$
Three remarks:

- **Back to the ability example.** With $\varepsilon_i=\gamma\,\text{abil}_i+v_i$, the last term is $\gamma\,\text{Cov}(z_i,\text{abil}_i)+\text{Cov}(z_i,v_i)$. OLS was stuck with $\gamma\,\text{Cov}(\text{educ}_i,\text{abil}_i)\neq0$; IV only needs $\text{Cov}(z_i,\text{abil}_i)=0$, i.e. proximity to a college unrelated to ability. We have not observed ability: we have chosen a variable that is not related to it.
- **The Wald estimator is a special case.** When $z$ is binary, $\widehat{\text{Cov}}(z,y)/\widehat{\text{Cov}}(z,x)$ simplifies *exactly* to $(\bar y_1-\bar y_0)/(\bar x_1-\bar x_0)$: the boxed formula is the comparison of Section 2.1, extended to any instrument.
- **OLS is a special case too.** Set $z_i=x_i$ and the formula becomes $\widehat{\text{Cov}}(x,y)/\widehat{\text{Var}}(x)$: OLS is IV in which $x$ serves as its own instrument, which is valid only if $x$ is itself exogenous.

##### 2.3 Reading the formula: first stage and reduced form

Divide the numerator and the denominator of the boxed formula by $\widehat{\text{Var}}(z)$. Each becomes a simple-regression slope you already know how to compute:
$$
\hat\beta_1^{IV} = \frac{\widehat{\text{Cov}}(z,y)/\widehat{\text{Var}}(z)}{\widehat{\text{Cov}}(z,x)/\widehat{\text{Var}}(z)} = \frac{\text{slope of } y \text{ on } z}{\text{slope of } x \text{ on } z} = \frac{\text{reduced form}}{\text{first stage}}.
$$

- The **first stage** (regress $x$ on $z$) measures how strongly the lever moves $x$. This is the relevance condition, in numbers.
- The **reduced form** (regress $y$ on $z$) measures how much $y$ moves when the lever moves. Under exogeneity, there is no endogeneity problem in this regression ($z$ is uncorrelated with $\varepsilon$), so it can be estimated by plain OLS.

This is precisely the ratio announced in Section 1.2: "effect of $z$ on $y$" over "effect of $z$ on $x$". In the Card data, with the controls of Section 4, the first stage gives $+0.464$ year of schooling for growing up near a college and the reduced form gives $+0.104$ in log wage, hence $\hat\beta_1^{IV}=0.104/0.464\approx0.223$. You will recover this number in Section 4.3.

A useful consequence: under the exclusion restriction, the reduced form is zero *if and only if* $\beta_1$ is zero (provided relevance holds). If $z$ moves $x$ but $y$ does not react at all, $x$ has no effect on $y$.

##### 2.4 What if the conditions fail?

Without assuming $\text{Cov}(z_i,\varepsilon_i)=0$, the same calculation gives, as $N\to\infty$:
$$
\hat\beta_1^{IV}\;\longrightarrow\;\beta_1+\frac{\text{Cov}(z_i,\varepsilon_i)}{\text{Cov}(z_i,x_i)}, \qquad\text{to compare with}\qquad \hat\beta_1^{OLS}\;\longrightarrow\;\beta_1+\frac{\text{Cov}(x_i,\varepsilon_i)}{\text{Var}(x_i)}.
$$
This formula shows how the two conditions interact:

- **If exogeneity fails**, IV is inconsistent, exactly like OLS: the bias does not vanish with $N$. IV does not get rid of the endogeneity problem; it moves it from $x$ to $z$, where we hope it is easier to argue away.
- **The violation is divided by the strength of the first stage.** With a strong instrument, a small violation of exogeneity produces a small bias. With a weak instrument, the same small violation is *magnified*. Example: suppose proximity to a college had a small direct effect of $2\%$ on wages (an urban wage premium not captured by the controls). With a first stage of $0.464$, the IV estimate would be off by $0.02/0.464\approx0.04$, i.e. 4 percentage points, half the size of the OLS estimate. With a first stage ten times weaker, the bias would be ten times larger. An imperfect instrument can therefore be *worse* than OLS.

##### 2.5 The price to pay: precision

IV uses only the part of the variation in $x$ that is driven by $z$, and throws away the rest. Less variation means less information, hence a larger variance. In the simple case, the variance of the IV estimator is approximately
$$
\text{Var}(\hat\beta_1^{IV})\approx\frac{\sigma^2}{N\,\text{Var}(x)}\cdot\frac{1}{\rho_{xz}^2},
$$
where $\rho_{xz}$ is the correlation between $x$ and $z$. The first factor is the OLS variance of Session 2; the second inflates it, and the inflation is severe when the instrument is weakly correlated with $x$. In the Card data, $\rho_{xz}\approx0.14$, so $1/\rho_{xz}^2\approx48$: the variance is multiplied by about 50, the standard error by about 7. In Section 4, the standard error on education goes from $0.004$ (OLS) to about $0.04$ (2SLS).

This is a **bias-variance trade-off**. OLS is precise but centred on the wrong value; IV is centred on the right value (if the instrument is valid) but noisy. With a large sample and a strong instrument, IV wins. With a small sample or a weak instrument, its confidence intervals can be so wide that they are uninformative.

---

# 3. Two-stage least squares (2SLS)

##### 3.1 Why we need a generalization

The ratio formula of Section 2 covers one endogenous regressor, one instrument, and nothing else. Real applications almost always need more:

- **Control variables.** In the Card data, workers who grew up near a college are more often from the North and from cities, where wages are higher regardless of schooling. Proximity is therefore a credible instrument only *conditional on* region and other characteristics: we need to include controls such as `exper`, `black` and `south` (Section 1.2 already noted that relevance and exogeneity should hold "after controlling for the other regressors").
- **Several instruments.** The Card data also contains `nearc2` (grew up near a *two-year* college). With two instruments, the ratio formula gives two different answers, $\widehat{\text{Cov}}(z_1,y)/\widehat{\text{Cov}}(z_1,x)$ and $\widehat{\text{Cov}}(z_2,y)/\widehat{\text{Cov}}(z_2,x)$. Which one should we use? Ideally, both at once.

Two-stage least squares (2SLS) handles both cases with a recipe that is, genuinely, just two ordinary regressions.

##### 3.2 The two stages

*Stage 1: separate the good variation from the bad*

Regress the endogenous regressor on the instrument(s) **and all the exogenous controls**, and keep the fitted values (literally `smf.ols(...).fittedvalues`, nothing new):
$$
\text{educ}_i = \pi_0+\pi_1\,\text{nearc4}_i+\pi_2\,\text{exper}_i+\pi_3\,\text{black}_i+\pi_4\,\text{south}_i + u_i \quad\Longrightarrow\quad \widehat{\text{educ}}_i.
$$
Like any OLS regression, this splits education into two pieces:

```
   educ_i   =        educ_hat_i          +          u_hat_i

               predicted from nearc4             everything else,
               and the controls only:            including the part driven
               uncorrelated with ε               by ability: correlated with ε

                     → KEPT                          → THROWN AWAY
```

This is literally the split between "good" and "bad" variation of Section 1.2. $\widehat{\text{educ}}_i$ is built only from variables that are uncorrelated with $\varepsilon_i$ (the instrument, by exogeneity; the controls, by assumption), so it is itself uncorrelated with $\varepsilon_i$. All the ability-driven variation in schooling ends up in the residual $\hat u_i$.

*Stage 2: use only the good variation*

Regress $y$ on $\widehat{\text{educ}}_i$ (in place of $\text{educ}_i$) and the same controls:
$$
\log w_i = \beta_0+\beta_1\,\widehat{\text{educ}}_i+\beta_2\,\text{exper}_i+\beta_3\,\text{black}_i+\beta_4\,\text{south}_i+\text{error}_i.
$$
$\hat\beta_1$ from this second regression is the **2SLS estimator**. Why does it work? Replacing $\text{educ}_i$ by $\widehat{\text{educ}}_i$ moves $\beta_1\hat u_i$ into the error term: $\text{error}_i=\varepsilon_i+\beta_1\hat u_i$. Neither piece is correlated with the regressors of stage 2: $\varepsilon_i$ because they are all exogenous, and $\hat u_i$ because first-stage residuals are, by construction, uncorrelated with the first-stage regressors. The condition that OLS needs, and that was violated in Section 1, is restored.

*Why the controls must appear in both stages*

The exogenous controls act as their own instruments: they must be in the first stage too. If `south` were left out of stage 1, the part of education explained by region would end up in $\hat u_i$, which would then be correlated with `south` in stage 2, and the argument above would collapse. Rule: **every exogenous variable of the second stage appears in the first stage.**

*Check: 2SLS is the IV formula of Section 2*

In the simple case (one regressor, one instrument, no controls), $\hat x_i=\hat\pi_0+\hat\pi_1z_i$ with $\hat\pi_1=\widehat{\text{Cov}}(z,x)/\widehat{\text{Var}}(z)$. The stage-2 slope is
$$
\frac{\widehat{\text{Cov}}(\hat x,y)}{\widehat{\text{Var}}(\hat x)} = \frac{\hat\pi_1\,\widehat{\text{Cov}}(z,y)}{\hat\pi_1^2\,\widehat{\text{Var}}(z)} = \frac{\widehat{\text{Cov}}(z,y)}{\hat\pi_1\,\widehat{\text{Var}}(z)} = \frac{\widehat{\text{Cov}}(z,y)}{\widehat{\text{Cov}}(z,x)}.
$$
2SLS and IV are numerically identical: 2SLS is simply the version that extends to controls and several instruments. In the Card data, both give $0.2234$.

*Several instruments*

With several instruments, nothing changes in the recipe: they all enter stage 1 as regressors, and $\hat x_i$ combines them into the best linear predictor of $x$, i.e. the strongest single lever that can be built from them. The only requirement is a counting one: you need **at least as many instruments as endogenous regressors**. With exactly as many, the model is *just identified* (the Card application); with more, it is *over-identified*, and you can test whether the instruments give compatible answers (overidentification test). Be careful with what this test says: if all instruments are invalid in the same way, they will agree with each other and the test will not detect anything. Exogeneity still has to be argued.

##### 3.3 Standard errors: do not trust the stage-2 output

If you run the two regressions by hand, as we do in Section 4, the *coefficient* from stage 2 is correct, but its reported standard error is not. The reason: stage 2 computes its residuals with $\widehat{\text{educ}}_i$, i.e. $\log w_i-\hat\beta_0-\hat\beta_1\widehat{\text{educ}}_i-\dots$, whereas the error of the model we care about involves the *actual* education, $\log w_i-\hat\beta_0-\hat\beta_1\text{educ}_i-\dots$ The two differ by $\hat\beta_1\hat u_i$, so the estimate of $\sigma$, and every standard error built on it, is wrong. In the Card data, the by-hand standard error is $0.036$ while the correct one is $0.041$: the naive output overstates precision by about 15%.

In practice, use a dedicated 2SLS routine, which runs both stages and computes the correct standard errors, for instance `linearmodels` (`pip install linearmodels`):
```python
from linearmodels.iv import IV2SLS
IV2SLS.from_formula('lwage ~ 1 + exper + black + south + [educ ~ nearc4]', data=card).fit()
```
The brackets read "`educ` is endogenous, instrumented by `nearc4`". Running the stages by hand remains the best way to *understand* what the routine does.

##### 3.4 Weak instruments

An instrument is **weak** when it moves $x$ only a little, once the controls are accounted for. Sections 2.4 and 2.5 already showed two consequences; there is a third:

1. **Imprecision.** We divide by a first-stage effect that is small and noisily estimated: the variance is inflated by $1/\rho_{xz}^2$ (Section 2.5), and confidence intervals become very wide.
2. **Magnified bias.** Any small violation of exogeneity is divided by the first stage (Section 2.4), and can make IV worse than OLS.
3. **Bias towards OLS, even with a valid instrument.** IV is consistent (right as $N\to\infty$), but not unbiased in finite samples. In a given sample, the first stage picks up, by chance, a bit of correlation between $z$ and the "bad" variation $u$, so $\hat x$ is not perfectly purged of ability. When the true effect of $z$ on $x$ is strong, this chance component is negligible. When it is weak, the chance component is a large share of $\hat x$, and 2SLS drifts back towards the biased OLS estimate it was supposed to correct.

*How to diagnose it*

Look at the first stage and test that the instrument coefficients are zero ($H_0:\pi_1=0$). The usual rule of thumb (Staiger and Stock, 1997) is a **first-stage $F$-statistic above 10**. With a single instrument, this $F$ is simply the square of the $t$-statistic on the instrument. Recent research argues that much higher thresholds are needed for reliable inference, so 10 is a minimum, not a comfort zone.

Be careful which $F$ you look at. The $F$-statistic printed at the top of a regression summary tests *all* the regressors jointly, controls included. In the Card first stage, that global $F$ is about $670$, mostly because experience and race strongly predict schooling, while the $F$ for `nearc4` alone is about $35$. Only the latter measures the strength of the instrument.

A weak instrument is, in practice, often worse than no instrument at all.

---

# 4. Application: does education really raise wages?

We use `card` (`wooldridge`): $3\,010$ men, 1976 wages and schooling, with `nearc4` ($=1$ if the respondent grew up near a four-year college) as the instrument discussed in Section 1.2.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from wooldridge import load_data

card = load_data.data('card')
card[['lwage', 'educ', 'nearc4', 'exper', 'black', 'south']].describe()


### 4.1 OLS first (the biased benchmark)

**Task.** Estimate $\log(\text{wage})=\beta_0+\beta_1\,\text{educ}+\beta_2\,\text{exper}+\beta_3\,\text{black}+\beta_4\,\text{south}+\varepsilon$ by OLS. $\hat\beta_1$ is (approximately) the % wage increase per extra year of schooling. Keep this number in mind for comparison below.

In [ ]:
ols = smf.ols('lwage ~ educ + exper + black + south', data=card).fit()
print(ols.summary())


### 4.2 Stage 1: does proximity to college predict schooling?

**Task.** Estimate the first stage, $\text{educ}=\pi_0+\pi_1\,\text{nearc4}+\pi_2\,\text{exper}+\pi_3\,\text{black}+\pi_4\,\text{south}+u$. Check relevance: is `nearc4` statistically significant? Is the first-stage $F$-statistic *for the instrument* comfortably above the rule-of-thumb threshold of $10$ mentioned in Section 3.4? (Not the global $F$ printed in the summary, which also tests the controls.)

In [ ]:
first_stage = smf.ols('educ ~ nearc4 + exper + black + south', data=card).fit()
print(first_stage.summary())
# With a single instrument, the F-statistic for H0: pi_1 = 0 is the square of its t-statistic
print("\nFirst-stage F-statistic (instrument only):", first_stage.tvalues['nearc4']**2)
print("Global F-statistic (all regressors, NOT the relevant one):", first_stage.fvalue)

### 4.3 Stage 2: 2SLS

**Task.** Complete the code below: store the fitted values from the first stage as `educ_hat`, then regress `lwage` on `educ_hat` (plus the same controls) for the second stage.

In [ ]:
# TODO: get the fitted values from the first-stage regression above
card2 = card.dropna(subset=['lwage', 'educ', 'nearc4', 'exper', 'black', 'south']).copy()
first_stage2 = smf.ols('educ ~ nearc4 + exper + black + south', data=card2).fit()
card2['educ_hat'] = ...

second_stage = smf.ols('lwage ~ educ_hat + exper + black + south', data=card2).fit()
print(second_stage.params[['educ_hat']])
assert np.isclose(second_stage.params['educ_hat'], 0.2234, atol=0.001), "Doesn't match the expected 2SLS coefficient - check educ_hat."
print("Self-check passed!")


**Task.** Compare the 2SLS coefficient on education to the OLS one from 4.1. This is a famous, genuinely surprising result in this literature: the IV estimate is *larger* than OLS, not smaller - the opposite of what a simple "ability bias" story alone would predict (more able people both study more *and* earn more, which should make OLS too *large*, not too small). Two of the leading explanations in the literature: **measurement error** in self-reported schooling (which biases OLS *toward* zero, partly offsetting ability bias), and the idea that IV estimates a *local* effect - the return to schooling specifically for the people whose decision was swayed by college proximity, who may have a particularly high return to the marginal year of schooling. Econometrics does not resolve this by itself; it tells you precisely what you are, and are not, estimating, so you can argue about it on economic grounds.

---

# 5. A short note: what are ANOVA and ANCOVA?

You may encounter these two names elsewhere - they are not new tools, just older vocabulary for something you already know.

**ANOVA** (analysis of variance) compares a numerical outcome $y$ across several categories - e.g. average sales across four marketing campaigns. It is taught, historically, as its own technique (splitting total variance into a "between-group" and a "within-group" part, Section 3-4 of Session 3's $R^2$ decomposition). But run it as a regression of $y$ on dummy variables for the categories, exactly as in Session 3, and the $F$-test for "do the group means differ?" is *identical* to ANOVA's $F$-test. ANOVA is OLS with only categorical regressors.

**ANCOVA** (analysis of covariance) is ANOVA plus one or more continuous **covariates** - e.g. comparing the four campaigns' average sales *while controlling for* store size. This is, again, nothing other than multivariate OLS with a mix of dummy and continuous regressors - precisely the kind of specification built throughout Session 3 (`lsalary ~ roe + lsales + finance`, for instance).

The only reason to know the two names is that you will meet them in other courses or software (R's `aov()`, SPSS menus) that present this as separate machinery. It is not: it is the regression toolkit you already have, applied to a dataset where the regressors happen to be categorical.

---

## Summary

| Step | Question | Tool |
|---|---|---|
| Section 1.1 | Why can't we just add more controls? | The confounder is unobserved and sits in $\varepsilon$: $\text{Cov}(x,\varepsilon)\neq0$, not fixed by a larger $N$ |
| Section 1.2 | What do we need instead? | An instrument $z$: relevant ($\text{Cov}(z,x)\neq0$) and exogenous ($\text{Cov}(z,\varepsilon)=0$) |
| Section 2 | The simplest case | $\hat\beta^{IV}=\widehat{\text{Cov}}(z,y)/\widehat{\text{Cov}}(z,x)$ = reduced form / first stage |
| Section 3 | The general case | Two-stage least squares: two ordinary regressions, chained together; check the first-stage $F$ of the instrument |
| Section 4 | Does it change the answer? | Yes, substantially: the return to education roughly triples with 2SLS ($0.078\to0.223$) |
| Section 5 | ANOVA/ANCOVA? | Already yours: OLS with categorical (+ continuous) regressors |

Practice exercises applying all of this are provided in the companion TD notebook.